# Forecasting Baselines in Colab

This notebook turns the current Prophet and XGBoost scaffolds into reproducible Colab experiment runs and also documents the first real Phase 2 public-source flow for HDC and NHSO.

Outputs produced by this notebook:

- `data/processed/hdc_ncd_prevalence.parquet`
- `data/processed/nhso_utilization.parquet`
- `data/processed/phase2_public_anchors.parquet`
- `data/processed/prophet_baseline_results_colab.json`
- `data/processed/prophet_baseline_results_colab.csv`
- `data/processed/xgboost_baseline_results_colab.json`
- `data/processed/xgboost_predictions_colab.csv`
- `data/processed/xgboost_feature_importance_colab.csv`
- `notebooks/results/02_forecasting_baselines_colab.zip`

The git write cells are intentionally guarded. Enable them only in a trusted environment with repository credentials configured.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/siriponsri/pharma-intelligence.git"
BRANCH = "step-a-improve-mock-data"
WORKDIR = Path("/content")
REPO_DIR = WORKDIR / "pharma-intelligence"

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[forecasting]"], check=True)

print(f"Repository ready at {REPO_DIR}")

## 1. Audit Repository Files and Detect Dead Assets

Use a lightweight audit first. The goal is to find empty files, cache directories, stale generated outputs, or duplicate artifacts before running experiments.

In [ ]:
import json
from collections import defaultdict

repo_paths = [path for path in REPO_DIR.rglob("*") if ".git" not in path.parts]
zero_byte_files = sorted(str(path.relative_to(REPO_DIR)) for path in repo_paths if path.is_file() and path.stat().st_size == 0)
cache_paths = sorted(
    str(path.relative_to(REPO_DIR))
    for path in repo_paths
    if any(part in {"__pycache__", ".ipynb_checkpoints"} for part in path.parts)
)
by_name = defaultdict(list)
for path in repo_paths:
    if path.is_file():
        by_name[path.name].append(str(path.relative_to(REPO_DIR)))
duplicate_basenames = {name: paths for name, paths in by_name.items() if len(paths) > 1}

print(json.dumps(
    {
        "zero_byte_files": zero_byte_files,
        "cache_paths": cache_paths,
        "duplicate_basenames": duplicate_basenames,
    },
    indent=2,
    ensure_ascii=False,
))

## 2. Remove Dead Files and Verify Project State

Only delete files after manual confirmation. Keep the list empty by default so the notebook does not remove anything unexpectedly.

In [ ]:
import shutil

CONFIRMED_DEAD_FILES: list[str] = []

for relative_path in CONFIRMED_DEAD_FILES:
    target_path = REPO_DIR / relative_path
    if target_path.is_file():
        target_path.unlink()
    elif target_path.is_dir():
        shutil.rmtree(target_path)

subprocess.run([sys.executable, "-m", "compileall", "src/pharma_intel", "scripts"], check=True)
subprocess.run(["git", "status", "--short"], check=True)
print("Cleanup review complete.")

## 3. Stage, Commit, and Push the Scaffold

This section is optional. It is safe by default because `ENABLE_GIT_WRITE` starts as `False`.

In [ ]:
ENABLE_GIT_WRITE = False
COMMIT_MESSAGE = "feat(infra): add phase 2 to 4 scaffold and experiment registry"

status = subprocess.run(["git", "status", "--short"], capture_output=True, text=True, check=True)
print(status.stdout or "Working tree clean.")

if ENABLE_GIT_WRITE:
    subprocess.run(["git", "add", "-A"], check=True)
    commit_result = subprocess.run(["git", "commit", "-m", COMMIT_MESSAGE], capture_output=True, text=True)
    print(commit_result.stdout)
    print(commit_result.stderr)
    subprocess.run(["git", "push", "origin", BRANCH], check=True)
else:
    print("Set ENABLE_GIT_WRITE = True only after configuring Colab git credentials.")

## 4. Load Configuration for HDC and NHSO

Provide real exported files from the public portals. The current project code normalizes those exports into stable project schemas.

If you store files in Google Drive, mount it first and point the input paths at your exported files.

In [ ]:
import numpy as np
import pandas as pd
import polars as pl

from pharma_intel.common import settings
from pharma_intel.forecasting import (
    GeneratorConfig,
    aggregate_to_class_monthly,
    evaluate_forecast,
    fit_prophet_baseline,
    fit_xgboost_global_model,
    generate_demand_panel,
    mape,
    prepare_training_frame,
    prophet_available,
    xgboost_available,
)
from pharma_intel.ingestion.hdc_scraper import run as run_hdc
from pharma_intel.ingestion.nhso_scraper import run as run_nhso

USE_GOOGLE_DRIVE = False
if USE_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    INPUT_ROOT = Path("/content/drive/MyDrive/pharma_intelligence_inputs")
else:
    INPUT_ROOT = Path("/content")

HDC_INPUT_PATH = None
NHSO_INPUT_PATH = None

settings.ensure_dirs()
for source_name in ("hdc", "nhso"):
    (settings.raw_dir / source_name).mkdir(parents=True, exist_ok=True)


def stage_raw_export(source_path: Path | None, target_dir: Path) -> Path | None:
    if source_path is None:
        return None
    source_path = Path(source_path)
    if not source_path.exists():
        raise FileNotFoundError(f"Input file not found: {source_path}")
    target_dir.mkdir(parents=True, exist_ok=True)
    target_path = target_dir / source_path.name
    if source_path.resolve() != target_path.resolve():
        target_path.write_bytes(source_path.read_bytes())
    return target_path


HDC_STAGED_PATH = stage_raw_export(HDC_INPUT_PATH, settings.raw_dir / "hdc")
NHSO_STAGED_PATH = stage_raw_export(NHSO_INPUT_PATH, settings.raw_dir / "nhso")

print({
    "repo_dir": str(REPO_DIR),
    "data_dir": str(settings.data_dir),
    "hdc_input_path": str(HDC_STAGED_PATH) if HDC_STAGED_PATH else None,
    "nhso_input_path": str(NHSO_STAGED_PATH) if NHSO_STAGED_PATH else None,
})

## 5. Ingest HDC Data End-to-End

Run the HDC normalization contract on a real exported file. If no file is available, this cell stops with a clear error rather than silently producing a misleading empty dataset.

In [ ]:
if HDC_STAGED_PATH is None and not any((settings.raw_dir / "hdc").glob("*")):
    raise FileNotFoundError("Provide an HDC export via HDC_INPUT_PATH or place one under data/raw/hdc/")

hdc_output_path = settings.processed_dir / "hdc_ncd_prevalence.parquet"
run_hdc(output_path=hdc_output_path, input_path=HDC_STAGED_PATH)
hdc_df = pl.read_parquet(hdc_output_path)
hdc_quality = json.loads((settings.interim_dir / "hdc_ncd_prevalence_quality.json").read_text(encoding="utf-8"))

print(hdc_quality)
hdc_df.head().to_pandas()

## 6. Ingest NHSO Data End-to-End

Run the NHSO normalization contract on a real exported file and persist the cleaned parquet plus quality report.

In [ ]:
if NHSO_STAGED_PATH is None and not any((settings.raw_dir / "nhso").glob("*")):
    raise FileNotFoundError("Provide an NHSO export via NHSO_INPUT_PATH or place one under data/raw/nhso/")

nhso_output_path = settings.processed_dir / "nhso_utilization.parquet"
run_nhso(output_path=nhso_output_path, input_path=NHSO_STAGED_PATH)
nhso_df = pl.read_parquet(nhso_output_path)
nhso_quality = json.loads((settings.interim_dir / "nhso_utilization_quality.json").read_text(encoding="utf-8"))

print(nhso_quality)
nhso_df.head().to_pandas()

## 7. Normalize Schemas and Build the Phase 2 Dataset

Map HDC and NHSO into one long-form public-anchor dataset that downstream experiments can inspect or enrich later.

In [ ]:
hdc_anchor = hdc_df.with_columns(
    [
        pl.col("year").cast(pl.Utf8).alias("period"),
        pl.concat_str([pl.col("age_group").fill_null("unknown"), pl.lit("|"), pl.col("gender").fill_null("unknown")]).alias("segment"),
        pl.col("disease_code").alias("concept_code"),
        pl.lit("disease_code").alias("concept_type"),
        pl.lit("prevalence_per_1000").alias("measure_name"),
        pl.col("prevalence_per_1000").alias("measure_value"),
        pl.lit("per_1000_population").alias("value_unit"),
        pl.lit("hdc").alias("source_id"),
        pl.col("province").alias("geography"),
    ]
).select([
    "source_id",
    "period",
    "geography",
    "segment",
    "concept_type",
    "concept_code",
    "measure_name",
    "measure_value",
    "value_unit",
])

nhso_anchor = nhso_df.with_columns(
    [
        pl.col("service_month").alias("period"),
        pl.col("scheme").alias("segment"),
        pl.col("drug_class").alias("concept_code"),
        pl.lit("drug_class").alias("concept_type"),
        pl.lit("claims_count").alias("measure_name"),
        pl.col("claims_count").cast(pl.Float64).alias("measure_value"),
        pl.lit("claims").alias("value_unit"),
        pl.lit("nhso").alias("source_id"),
        pl.lit(None, dtype=pl.Utf8).alias("geography"),
    ]
).select(hdc_anchor.columns)

phase2_public_anchors = pl.concat([hdc_anchor, nhso_anchor], how="vertical")
phase2_output_path = settings.processed_dir / "phase2_public_anchors.parquet"
phase2_public_anchors.write_parquet(phase2_output_path)

print({"rows": phase2_public_anchors.height, "output_path": str(phase2_output_path)})
phase2_public_anchors.head().to_pandas()

## 8. Validate the End-to-End Pipeline

Run focused checks on row counts, uniqueness, null levels, and artifact paths.

In [ ]:
assert hdc_df.height > 0, "HDC dataset is empty"
assert nhso_df.height > 0, "NHSO dataset is empty"
assert phase2_public_anchors.height == hdc_anchor.height + nhso_anchor.height
assert phase2_public_anchors.select(pl.col("measure_value").is_null().sum()).item() == 0
assert phase2_public_anchors.select(pl.len()).item() > 0

validation_summary = {
    "hdc_rows": hdc_df.height,
    "nhso_rows": nhso_df.height,
    "phase2_rows": phase2_public_anchors.height,
    "phase2_duplicates": phase2_public_anchors.is_duplicated().sum(),
    "phase2_output_exists": phase2_output_path.exists(),
}
print(validation_summary)
phase2_public_anchors.group_by(["source_id", "measure_name"]).len().sort(["source_id", "measure_name"]).to_pandas()

## 9. Sync Required Experiment Files to the Repository

Confirm that the experiment docs exist and that the forecasting input panel is available. Generate the synthetic panel only when it is missing.

In [ ]:
required_docs = [
    REPO_DIR / "docs" / "experiment_08_prophet.md",
    REPO_DIR / "docs" / "experiment_09_xgboost.md",
]
for doc_path in required_docs:
    if not doc_path.exists():
        raise FileNotFoundError(f"Missing experiment doc: {doc_path}")

mock_parquet_path = settings.processed_dir / "mock_demand_history.parquet"
mock_csv_path = settings.processed_dir / "mock_demand_history.csv"
if not mock_parquet_path.exists():
    demand_panel = generate_demand_panel(config=GeneratorConfig(random_seed=42))
    demand_panel.write_parquet(mock_parquet_path)
    demand_panel.write_csv(mock_csv_path)
else:
    demand_panel = pl.read_parquet(mock_parquet_path)

class_month_df = aggregate_to_class_monthly(demand_panel)
print({
    "mock_parquet_path": str(mock_parquet_path),
    "mock_rows": demand_panel.height,
    "class_month_rows": class_month_df.height,
})

## 10. Run a Full Prophet Experiment in Colab

Run Prophet twice: endogenous only and with exogenous features. Save per-class metrics and a compact summary.

In [ ]:
if not prophet_available():
    raise RuntimeError("Prophet is not installed. Re-run the install cell.")

PROPHET_EXOG_FEATURES = [
    "diabetes_prevalence",
    "hypertension_prevalence",
    "dyslipidemia_prevalence",
    "obesity_prevalence",
    "healthcare_budget_index",
    "stockout_days",
]
TEST_MONTHS = 12


def summarize_metric_frame(frame: pl.DataFrame, variant: str) -> dict:
    return {
        "variant": variant,
        "n_classes": frame.height,
        "mean_mape": round(frame["mape"].mean(), 3),
        "median_mape": round(frame["mape"].median(), 3),
        "mean_smape": round(frame["smape"].mean(), 3),
        "mean_rmse": round(frame["rmse"].mean(), 3),
        "mean_mase": round(frame["mase"].mean(), 3),
    }


def run_prophet_experiment(source_df: pl.DataFrame, with_exog: bool) -> tuple[pl.DataFrame, dict]:
    rows: list[dict] = []
    exog_cols = PROPHET_EXOG_FEATURES if with_exog else None
    for class_id in sorted(source_df["class_id"].unique().to_list()):
        series = source_df.filter(pl.col("class_id") == class_id).sort("month")
        train = series.head(len(series) - TEST_MONTHS)
        test = series.tail(TEST_MONTHS)
        forecast = fit_prophet_baseline(train, exog_cols=exog_cols, horizon_months=TEST_MONTHS)
        metrics = evaluate_forecast(
            test["units_kddd"].to_list(),
            forecast,
            y_train=train["units_kddd"].to_list(),
        )
        rows.append(
            {
                "variant": "prophet_exog" if with_exog else "prophet",
                "class_id": class_id,
                **{metric_name: round(metric_value, 3) for metric_name, metric_value in metrics.items()},
            }
        )
    result_frame = pl.DataFrame(rows).sort("mape")
    summary = summarize_metric_frame(result_frame, "prophet_exog" if with_exog else "prophet")
    return result_frame, summary


prophet_base_df, prophet_base_summary = run_prophet_experiment(class_month_df, with_exog=False)
prophet_exog_df, prophet_exog_summary = run_prophet_experiment(class_month_df, with_exog=True)
prophet_all_df = pl.concat([prophet_base_df, prophet_exog_df], how="vertical")

prophet_json_path = settings.processed_dir / "prophet_baseline_results_colab.json"
prophet_csv_path = settings.processed_dir / "prophet_baseline_results_colab.csv"
prophet_payload = {
    "random_seed": 42,
    "test_months": TEST_MONTHS,
    "summaries": [prophet_base_summary, prophet_exog_summary],
    "rows": prophet_all_df.to_dicts(),
}
prophet_json_path.write_text(json.dumps(prophet_payload, indent=2, ensure_ascii=False), encoding="utf-8")
prophet_all_df.write_csv(prophet_csv_path)

pd.DataFrame([prophet_base_summary, prophet_exog_summary]), prophet_all_df.to_pandas().head(20)

## 11. Run a Full XGBoost Experiment in Colab

Train the global XGBoost baseline on engineered class-month features with a last-12-month holdout and export both summary metrics and row-level predictions.

In [ ]:
if not xgboost_available():
    raise RuntimeError("XGBoost is not installed. Re-run the install cell.")

train_frame = prepare_training_frame(class_month_df)
latest_months = sorted(train_frame["month"].unique().to_list())[-TEST_MONTHS:]
train_split = train_frame.filter(~pl.col("month").is_in(latest_months))
test_split = train_frame.filter(pl.col("month").is_in(latest_months))

xgb_artifact = fit_xgboost_global_model(train_split)
feature_columns = xgb_artifact["feature_columns"]
model = xgb_artifact["model"]

xgb_test_features = test_split.select(
    [pl.col(column_name).cast(pl.Float64, strict=False).fill_null(0.0).alias(column_name) for column_name in feature_columns]
)
predictions = model.predict(xgb_test_features.to_numpy())

xgb_prediction_frame = test_split.select(["month", "class_id", "therapeutic_area", "units_kddd"]).with_columns(
    pl.Series("prediction", predictions),
    (pl.col("units_kddd") - pl.Series("prediction", predictions)).abs().alias("absolute_error"),
)
actual = xgb_prediction_frame["units_kddd"].to_numpy()

xgb_overall_summary = {
    "training_rows": xgb_artifact["n_rows"],
    "feature_count": len(feature_columns),
    "held_out_rows": test_split.height,
    "mae": round(float(np.mean(np.abs(actual - predictions))), 3),
    "rmse": round(float(np.sqrt(np.mean((actual - predictions) ** 2))), 3),
    "mape": round(float(mape(actual, predictions)), 3),
}

xgb_predictions_pdf = xgb_prediction_frame.to_pandas()
per_class_rows: list[dict] = []
for class_id, group in xgb_predictions_pdf.groupby("class_id"):
    class_actual = group["units_kddd"].to_numpy()
    class_pred = group["prediction"].to_numpy()
    per_class_rows.append(
        {
            "class_id": class_id,
            "n_rows": int(len(group)),
            "mae": round(float(np.mean(np.abs(class_actual - class_pred))), 3),
            "rmse": round(float(np.sqrt(np.mean((class_actual - class_pred) ** 2))), 3),
            "mape": round(float(mape(class_actual, class_pred)), 3),
        }
    )

xgb_per_class = pl.DataFrame(per_class_rows).sort("mae")
feature_importance_df = pd.DataFrame(
    {
        "feature": feature_columns,
        "importance": model.feature_importances_,
    }
).sort_values("importance", ascending=False)

xgb_summary_path = settings.processed_dir / "xgboost_baseline_results_colab.json"
xgb_predictions_path = settings.processed_dir / "xgboost_predictions_colab.csv"
xgb_importance_path = settings.processed_dir / "xgboost_feature_importance_colab.csv"

xgb_summary_path.write_text(
    json.dumps(
        {
            "random_seed": 42,
            "test_months": TEST_MONTHS,
            "summary": xgb_overall_summary,
            "per_class": xgb_per_class.to_dicts(),
        },
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)
xgb_prediction_frame.write_csv(xgb_predictions_path)
feature_importance_df.to_csv(xgb_importance_path, index=False)

pd.DataFrame([xgb_overall_summary]), xgb_per_class.to_pandas().head(20), feature_importance_df.head(20)

## 12. Persist Metrics, Artifacts, and Predictions

Bundle the main artifacts so the run can be downloaded from Colab and copied back into the repository if needed.

In [ ]:
from zipfile import ZipFile

artifact_paths = [
    hdc_output_path,
    nhso_output_path,
    phase2_output_path,
    settings.interim_dir / "hdc_ncd_prevalence_quality.json",
    settings.interim_dir / "nhso_utilization_quality.json",
    prophet_json_path,
    prophet_csv_path,
    xgb_summary_path,
    xgb_predictions_path,
    xgb_importance_path,
]

zip_output_path = REPO_DIR / "notebooks" / "results" / "02_forecasting_baselines_colab.zip"
zip_output_path.parent.mkdir(parents=True, exist_ok=True)
with ZipFile(zip_output_path, "w") as archive:
    for artifact_path in artifact_paths:
        archive.write(artifact_path, artifact_path.relative_to(REPO_DIR))

print({
    "zip_output_path": str(zip_output_path),
    "artifact_count": len(artifact_paths),
})

## Next Docs To Update

After running the notebook with real inputs, update these documents with the produced numbers and artifact paths:

- `docs/experiment_08_prophet.md`
- `docs/experiment_09_xgboost.md`
- `docs/experiment_04_real_data_integration.md`

Preserve the measured results exactly as produced by the notebook.